In [7]:
# github_data_extractor.py

import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

# ------------------------------
# Configuration
# ------------------------------
GITHUB_PAT = "ghp_xxx"  # Replace with your token
HEADERS = {"Authorization": f"token {GITHUB_PAT}"}
BASE_URL = "https://api.github.com"
TOPIC = "feathersjs"

# ------------------------------
# Helpers
# ------------------------------
def github_api(url):
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 202:
        return None
    response.raise_for_status()
    return response.json()

def retry_github_stats(url, retries=5, delay=5):
    for attempt in range(retries):
        result = github_api(url)
        if result is not None:
            return result
        time.sleep(delay)
    return []

def get_feathersjs_repos():
    repos = []
    for page in range(1, 5):  # Should cover 304 repos
        url = f"{BASE_URL}/search/repositories?q=topic:{TOPIC}&per_page=100&page={page}"
        data = github_api(url)
        if 'items' in data:
            repos.extend(data['items'])
        else:
            break
    return [(repo['owner']['login'], repo['name']) for repo in repos]

def get_repo_contributors(owner, repo):
    url = f"{BASE_URL}/repos/{owner}/{repo}/stats/contributors"
    return retry_github_stats(url)

def get_repo_commits(owner, repo):
    url = f"{BASE_URL}/repos/{owner}/{repo}/commits?per_page=100"
    commits = []
    while url:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        commits.extend(response.json())
        url = response.links.get('next', {}).get('url')
    return commits

def get_repo_punch_card(owner, repo):
    url = f"{BASE_URL}/repos/{owner}/{repo}/stats/punch_card"
    return retry_github_stats(url)

def get_repo_issues(owner, repo):
    url = f"{BASE_URL}/repos/{owner}/{repo}/issues?state=all&per_page=100"
    issues = []
    while url:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        issues.extend(response.json())
        url = response.links.get('next', {}).get('url')
    return issues

def get_repo_pull_requests(owner, repo):
    url = f"{BASE_URL}/repos/{owner}/{repo}/pulls?state=all&per_page=100"
    prs = []
    while url:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        prs.extend(response.json())
        url = response.links.get('next', {}).get('url')
    return prs

# ------------------------------
# Run ETL
# ------------------------------
if __name__ == "__main__":
    all_commits = []
    all_contributors = []
    all_punch_cards = []
    all_prs = []
    all_issues = []

    repos = get_feathersjs_repos()

    for owner, repo in tqdm(repos, desc="Processing Repositories"):
        print(f"\n🔄 {owner}/{repo}")
        try:
            all_commits.extend(get_repo_commits(owner, repo))
            all_contributors.extend(get_repo_contributors(owner, repo))
            all_punch_cards.extend(get_repo_punch_card(owner, repo))
            all_prs.extend(get_repo_pull_requests(owner, repo))
            all_issues.extend(get_repo_issues(owner, repo))
        except Exception as e:
            print(f"❌ Error processing {owner}/{repo}: {e}")

    # Convert to DataFrames
    df_commits = pd.json_normalize(all_commits)
    df_contributors = pd.json_normalize(all_contributors)
    df_punch_cards = pd.DataFrame(all_punch_cards, columns=["day", "hour", "commits"])
    df_prs = pd.json_normalize(all_prs)
    df_issues = pd.json_normalize(all_issues)

    # Save to disk
    df_commits.to_json("commits.json", indent=2)
    df_contributors.to_json("contributors.json", indent=2)
    df_punch_cards.to_json("punch_card.json", indent=2)
    df_prs.to_json("pull_requests.json", indent=2)
    df_issues.to_json("issues.json", indent=2)

    print("\n✅ All data extracted, transformed, and saved.")

Processing Repositories:   0%|                          | 0/296 [00:00<?, ?it/s]


🔄 LeCoupa/awesome-cheatsheets


Processing Repositories:   0%|                | 1/296 [00:29<2:26:13, 29.74s/it]


🔄 feathersjs/feathers


Processing Repositories:   1%|                | 2/296 [02:02<5:27:17, 66.79s/it]

...


🔄 ched-dev/places-api-dev


Processing Repositories: 100%|█████████████▉| 295/296 [1:04:47<00:11, 11.66s/it]


🔄 jruipinto/file-upload-with-multer-feathers-ws-example


Processing Repositories: 100%|██████████████| 296/296 [1:04:54<00:00, 13.16s/it]



✅ All data extracted, transformed, and saved.
